# Import the Dataframe

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
from shapely.geometry import Point
from scipy.stats import linregress

from particles_path import *

Read in the cleaned dataframe from the other notebook. 

**Reminder:** Our dataframe has 8352 data points and 32 variables per data point.

In [ ]:
clean_rad = pd.read_excel("cleaned_data.xlsx")
clean_rad

Create some subsets of the data based on the 5 major latitude zones so that we can observe trends between the different zones later.

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(clean_rad["lon"], clean_rad["lat"])]
geo_rad = gpd.GeoDataFrame(clean_rad, geometry=geometry, crs="EPSG:4326")

In [ ]:
df_arctic = clean_rad[clean_rad["lat"] > 66.5]
df_antarctic = clean_rad[clean_rad["lat"] < -66.5]
df_tropical = clean_rad[clean_rad["lat"].between(-23.5, 23.5)]
df_stz = clean_rad[clean_rad["lat"].between(-66.5, -23.5, inclusive="neither")]
df_ntz = clean_rad[clean_rad["lat"].between(23.5, 66.5, inclusive="neither")]

# Preliminary Findings

## Heatmaps (Full Data)

In [ ]:
cols = ["proton0_ps", "electron0_ps", "xray0_ps", "total_radiation_ps"]

fig, axes = plt.subplots(2, 2, figsize=(10, 5), constrained_layout=True)

axes = axes.flatten()

for i, col in enumerate(cols):
    world.plot(ax=axes[i], color="lightgrey")
    geo_rad.plot(ax=axes[i],
                markersize=1,
                alpha=0.5,
                column=col, 
                cmap="managua",
                legend=True
                )
    axes[i].set_title(f"Variable: {col}")

for ax in axes:  # Fix: loop over each axis
    ax.set_aspect("equal", adjustable="box")

fig.suptitle("All Main Standardized Variables")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))

world.plot(ax=ax1, color="lightgray")
sc1 = ax1.scatter(
    clean_rad["lon"],
    clean_rad["lat"],
    c=clean_rad["proton0_ps"],
    s=5,
    cmap="viridis"   # optional, but recommended
)
ax1.set_title("Proton Radiation per Second", fontsize="xx-large")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
ax1.set_aspect("equal")
cbar1 = fig.colorbar(sc1, ax=ax1, shrink=0.4)

world.plot(ax=ax2, color="lightgray")
sc2 = ax2.scatter(
    clean_rad["lon"],
    clean_rad["lat"],
    c=clean_rad["proton0_ps"],
    s=5,
    cmap="viridis",
    norm=LogNorm()
)
ax2.set_title("Proton Radiation per Second (log scale)", fontsize="xx-large")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
ax2.set_aspect("equal")
cbar2 = fig.colorbar(sc2, ax=ax2, shrink=0.4)

plt.tight_layout()
plt.show()

Heatmap for the hotspots of proton radiation per second. 

The linear scale makes it difficult to see trends in the high values due to the dataset containing a number of high outliers that throws off the scale and makes most datapoint that deep purple. Thus we made another one on a log scale to have a better view over where the bright values are. We can see the hotspots of proton radiation is the very top of the graph (Arctic Circle) and the Ocean area southeast of South America (South Atlantic Anomaly). 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))

world.plot(ax=ax1, color="lightgray")
sc1 = ax1.scatter(
    clean_rad["lon"],
    clean_rad["lat"],
    c=clean_rad["electron0_ps"],
    s=5,
    cmap="viridis"   # optional, but recommended
)
ax1.set_title("Electron Radiation per Second", fontsize="xx-large")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
ax1.set_aspect("equal")
cbar1 = fig.colorbar(sc1, ax=ax1, shrink=0.4)

world.plot(ax=ax2, color="lightgray")
sc2 = ax2.scatter(
    clean_rad["lon"],
    clean_rad["lat"],
    c=clean_rad["electron0_ps"],
    s=5,
    cmap="viridis",
    norm=LogNorm()
)
ax2.set_title("Electron Radiation per Second (log scale)", fontsize="xx-large")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
ax2.set_aspect("equal")
cbar2 = fig.colorbar(sc2, ax=ax2, shrink=0.4)

plt.tight_layout()
plt.show()

Heatmap for the hotspots of electron radiation per second. 

The linear scale does not give much information. The log scale gives a better view over where the higher values are. We can see that the hotspots of electron radiation is the same as the proton radiation heatmap, which is the very top of the graph (Arctic Circle) and the Ocean area southeast of South America (South Atlantic Anomaly). 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))

world.plot(ax=ax1, color="lightgray")
sc1 = ax1.scatter(
    clean_rad["lon"],
    clean_rad["lat"],
    c=clean_rad["xray0_ps"],
    s=5,
    cmap="viridis"   # optional, but recommended
)
ax1.set_title("X-Ray Radiation per Second", fontsize="xx-large")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
ax1.set_aspect("equal")
cbar1 = fig.colorbar(sc1, ax=ax1, shrink=0.4)

world.plot(ax=ax2, color="lightgray")
sc2 = ax2.scatter(
    clean_rad["lon"],
    clean_rad["lat"],
    c=clean_rad["xray0_ps"],
    s=5,
    cmap="viridis",
    norm=LogNorm()
)
ax2.set_title("X-Ray Radiation per Second (log scale)", fontsize="xx-large")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
ax2.set_aspect("equal")
cbar2 = fig.colorbar(sc2, ax=ax2, shrink=0.4)

plt.tight_layout()
plt.show()

These x-ray radiation heatmaps were the most interesting of the three types of radiation. This is due to the fact that even on the linear scale we can see the hotspot at the top of the grpah in the Arctic Circle. Then when we go to the log scale, it looks even more like we have a trend in the amount of x-ray radiation and the latitude of the satellite. 

### Analysis of particles per second

Comparing all of the low-threshold sensors standardized by second shows some interesting variation. 
- It looks as though, at these sensor thresholds, proton and, to a lesser extent, electeron radiation are higher the further south go. 
- Inversely, x-ray appears to get higher the further north you go. 
    - Does the season have an effect on this stark difference?
    - Is the difference because of hemispheric differece?
- Total radiation per second seems to be fairly steady with the main variation being sample size difference between the arctic and antarctic regions. 
    - The lack of overlap between high proton and electron regions and high xray regions might be why total radiation seems so steady.
    - Interestingly, there is some overlap between the detectors over Asia
- Barring one tiny point in the bottom left of the map, ses_ps is consistently low across the entire planet. That point might be worth researching.

## Heatmaps (Full Data Split By Month)

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr"]
cols = ["proton0_ps", "electron0_ps", "xray0_ps", "ses_ps", "total_radiation_ps"]

vmin = geo_rad[cols[:4]].min().min()
vmax = geo_rad[cols[:4]].max().max()

for col in cols:
    fig, axes = plt.subplots(2, 2, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    if col == "total_radiation_ps":
        vmin = geo_rad[col].min()
        vmax = geo_rad[col].max()

    for i, month in enumerate(months):
        subset = geo_rad[geo_rad["month"] == month]
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(
            ax=axes[i],
            markersize=1,
            alpha=0.5,
            column=col,
            cmap="managua",
            legend=False,
            vmin=vmin,
            vmax=vmax
        )
        axes[i].set_title(f"Variable: {month}")
        axes[i].set_xlabel("Longitude")
        axes[i].set_ylabel("Latitude")
       
    # Create a colorbar with no alha value applied
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    sm = cm.ScalarMappable(cmap="managua", norm=norm)
    sm.set_array([])  # required but unused

    plt.colorbar(sm, ax=axes, label=f"{col[:-3].title().replace("_", " ")} Per Second")
    fig.suptitle(f"{col[:-3].title().replace("_", " ")} Per Second by Month", va="top", fontsize="xx-large")
    plt.show()

#### Find the number of data points from each month

In [ ]:
clean_rad.groupby("month", sort=False)["month"].count()

#### Find and view all data points are in the southern hemisphere in March

In [ ]:
print(f"Data points in the entire southern hemisphere in March: {len(clean_rad[(clean_rad["lat"] < 0) & (clean_rad["month"] == "Mar")])}")
print(f"Data points in or below sourthern Temperate Zone in March: {len(clean_rad[(clean_rad["lat"] < -23.5) & (clean_rad["month"] == "Mar")])}")
print(f"Data points in the Antarctic Circle in March: {len(clean_rad[(clean_rad["lat"] < -66.5) & (clean_rad["month"] == "Mar")])}")

In [ ]:
clean_rad[(clean_rad["lat"] < -23.5) & (clean_rad["month"] == "Mar")][["timestamp", "packetID", "sample", "lat", "lon", 
                                                                       "electron0_ps", "proton0_ps", "xray0_ps", "ses_ps",
                                                                       "total_radiation_ps"]]

### Analysis of monthly splitting

#### Splitting these columns by month tells an interesting story:
- Protons stay pretty low for most of the time, but packets of higher counts are more common in January and February and in the southern hemisphere
- Electrons stay fairly steady across the globe, but the few values that get higher are pretty much all in January and February
- As the year progresses from January to April, the X-rays appear to become stronger in the Arctic Circle
    - This seems to correlate with when the sun should rise in the Arctic Circle
        - It might be worth splitting March by week to see if it grows on a weekly basis
    - X-rays stay similarly low in the southern hemisphere over all four months
- Ses basically never varies, and, when it does, it's not by much
    - There is one spike in April near lon -150 lat 60. It might be worth investigating what may have caused that, but Ses is not our focus for this project.
        - Without this one data point, ses's scale caps out at a much lower value
- While we are aware total radiation is not the most useful metric, it might still deliver some good insight.
    - Because of how little overlap between high values in our data, this column shows off every place where `proton0_ps`, `electron0_ps`, and `xray0_ps` spike

#### This brings up a few issues:
- We have almost no data in the southern hemisphere in March
    - In the entire southern hemisphere, we only have 32 data points total
    - Of the 32 data points, only have 17 data points within the South Temperate Zone
        - That accounts for one full packet plus the very first entry of another packet
    - Interestingly, we have more than double the data points in March than any other month despite no data in the south
- We have significantly fewer data points in April
    - This makes sense given our data stops on April 10th
- Only having three and a quarter months of data makes it really hard to make claims relating to seasonality
- Our sample sizes vary wildly based on location and month, making analysis difficult

## Heatmaps (Arctic Circle)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 4))

sc1 = ax1.scatter(
    df_arctic["lon"],
    df_arctic["lat"],
    c=df_arctic["proton0_ps"],
    s=5
)
ax1.set_title("Proton per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_arctic["lon"],
    df_arctic["lat"],
    c=df_arctic["proton0_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("Proton per Second (Log Scale)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 4))

sc1 = ax1.scatter(
    df_arctic["lon"],
    df_arctic["lat"],
    c=df_arctic["electron0_ps"],
    s=5
)
ax1.set_title("Electron per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_arctic["lon"],
    df_arctic["lat"],
    c=df_arctic["electron0_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("Electron per Second (Log Scale)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 4))

sc1 = ax1.scatter(
    df_arctic["lon"],
    df_arctic["lat"],
    c=df_arctic["xray0_ps"],
    s=5
)
ax1.set_title("X-Ray per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_arctic["lon"],
    df_arctic["lat"],
    c=df_arctic["xray0_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("X-Ray per Second (Log Scale)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

plt.tight_layout()
plt.show()

None of these three heatmaps were particulary beneficial in presenting new information to us that we could not see in the full dataset heatmap.

## Heatmaps (Antarctic Circle)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 4))

sc1 = ax1.scatter(
    df_antarctic["lon"],
    df_antarctic["lat"],
    c=df_antarctic["proton0_ps"],
    s=5
)
ax1.set_title("Proton per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_antarctic["lon"],
    df_antarctic["lat"],
    c=df_antarctic["proton0_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("Proton per Second (Log Scale)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 4))

sc1 = ax1.scatter(
    df_antarctic["lon"],
    df_antarctic["lat"],
    c=df_antarctic["electron0_ps"],
    s=5
)
ax1.set_title("Electron per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_antarctic["lon"],
    df_antarctic["lat"],
    c=df_antarctic["electron0_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("Electron per Second (Log Scale)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 4))

sc1 = ax1.scatter(
    df_antarctic["lon"],
    df_antarctic["lat"],
    c=df_antarctic["xray0_ps"],
    s=5
)
ax1.set_title("X-Ray per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_antarctic["lon"],
    df_antarctic["lat"],
    c=df_antarctic["xray0_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("X-Ray per Second (Log Scale)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

plt.tight_layout()
plt.show()

None of these three heatmaps were particulary beneficial in presenting new information to us that we could not see in the full dataset heatmap.

## Box and Whisker Plots (Across Latitude Zones)

In [ ]:
plt.figure(figsize=(15, 6))

plt.boxplot([
    df_arctic['total_radiation_ps'],
    df_ntz['total_radiation_ps'],
    df_tropical['total_radiation_ps'],
    df_stz['total_radiation_ps'],
    df_antarctic['total_radiation_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5],
    ['Arctic Circle','Northern Tropical Zone','Tropical Zone','Southern Tropical Zone','Antarctic Circle']
)

plt.yscale('log')
plt.ylabel('Count (Log Scale)')
plt.title('Radiation Distributions (per second) Across Major Latitude Zones')
plt.show()

We can see that the total radiation per second counts drops significantly in the tropical zone (around the equator). We initially thought that this was interesting, but after bringing this to NearSpace Launch and Dr. Voss, they explained that this is not new information and something that the community was already aware of. The magnetic field is strongest around the equator causing less radiation to enter the atmosphere. 

In [ ]:
plt.figure(figsize=(15, 6))

plt.boxplot([
    df_arctic['xray0_ps'],
    df_ntz['xray0_ps'],
    df_tropical['xray0_ps'],
    df_stz['xray0_ps'],
    df_antarctic['xray0_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5],
    ['Arctic Circle','Northern Tropical Zone','Tropical Zone','Southern Tropical Zone','Antarctic Circle']
)

plt.yscale('log')
plt.ylabel('Count (Log Scale)')
plt.title('X-Ray Radiation Distributions (per second) Across Major Latitude Zones')
plt.show()

This graph reinforces the trend we saw in our original heatmap, which is that x-ray radiation is much higher in the northern hemisphere than it is in the southern hemisphere. 

In [ ]:
plt.figure(figsize=(15, 6))

plt.boxplot([
    df_arctic['proton0_ps'],
    df_antarctic['proton0_ps'],
    df_arctic['electron0_ps'],
    df_antarctic['electron0_ps'],
    df_arctic['xray0_ps'],
    df_antarctic['xray0_ps'],
    df_arctic['total_radiation_ps'],
    df_antarctic['total_radiation_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5, 6, 7, 8],
    ['Proton Arctic', 'Proton Antarctic', 'Electron Arctic', 'Electron Antarctic', 
     'Xray Arctic', 'Xray Antarctic', 'Total Rad. Arctic', 'Total Rad. Antarctic']
)

plt.yscale('log')
plt.ylabel('Count (Log Scale)')
plt.title('Radiation Distributions (per second) in the Poles')
plt.show()

This graph compares the 3 types of radiations in the Arctic Circle versus the Antarctic Circle. We can see that the proton and electron radiation is higher in the Antarctic Circle while the x-ray radiation is higher in the Arctic Circle. However, when we look at the total radiation across the two poles, we can see that the total radiation amount is equal in both circles. 

## Line Graphs (Arctic Circle Groups)

Found which groups within the Arctic Circle contain the most amount of data. Then we created a subset of the Arctic dataset in order to plot the movements of the radiation types within the group. 

In [ ]:
df_arctic["group"].value_counts().head(5)

In [ ]:
df_arctic179 = df_arctic[df_arctic["group"]==179]
df_arctic189 = df_arctic[df_arctic["group"]==189]
df_arctic193 = df_arctic[df_arctic["group"]==193]
df_arctic159 = df_arctic[df_arctic["group"]==159]
df_arctic205 = df_arctic[df_arctic["group"]==205]

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_arctic179["timestamp"], df_arctic179["proton0"], label="Proton",color="blue")
ax1.plot(df_arctic179["timestamp"], df_arctic179["electron0"], label="Electon",color="green")
ax1.plot(df_arctic179["timestamp"], df_arctic179["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_arctic179["timestamp"], df_arctic179["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

date_form = mdates.DateFormatter('%H:%M:%S')
ax1.xaxis.set_major_formatter(date_form)
ax2.xaxis.set_major_formatter(date_form)

ax1.legend(lines1 + lines2, labels1 + labels2)
fig.supxlabel("Timestamp")
plt.title("Group 179 Data", fontsize="xx-large")
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_arctic189["timestamp"], df_arctic189["proton0"], label="Proton",color="blue")
ax1.plot(df_arctic189["timestamp"], df_arctic189["electron0"], label="Electon",color="green")
ax1.plot(df_arctic189["timestamp"], df_arctic189["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_arctic189["timestamp"], df_arctic189["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

date_form = mdates.DateFormatter('%H:%M:%S')
ax1.xaxis.set_major_formatter(date_form)
ax2.xaxis.set_major_formatter(date_form)

ax1.legend(lines1 + lines2, labels1 + labels2)
fig.supxlabel("Timestamp")
plt.title("Group 189 Data (Arctic Circle)", fontsize="xx-large")
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_arctic193["timestamp"], df_arctic193["proton0"], label="Proton",color="blue")
ax1.plot(df_arctic193["timestamp"], df_arctic193["electron0"], label="Electon",color="green")
ax1.plot(df_arctic193["timestamp"], df_arctic193["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_arctic193["timestamp"], df_arctic193["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2)
plt.title("Group 193 Data")
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_arctic159["timestamp"], df_arctic159["proton0"], label="Proton",color="blue")
ax1.plot(df_arctic159["timestamp"], df_arctic159["electron0"], label="Electon",color="green")
ax1.plot(df_arctic159["timestamp"], df_arctic159["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_arctic159["timestamp"], df_arctic159["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2)
plt.title("Group 159 Data")
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_arctic205["timestamp"], df_arctic205["proton0"], label="Proton",color="blue")
ax1.plot(df_arctic205["timestamp"], df_arctic205["electron0"], label="Electon",color="green")
ax1.plot(df_arctic205["timestamp"], df_arctic205["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_arctic205["timestamp"], df_arctic205["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2)
plt.title("Group 205 Data")
plt.show()

In the Arctic Circle, we can see a trend form across all 5 of the largest groups. That trend is that x-ray radiation is higher than proton radiation, and proton radiation is higher than electron radiation. These three radiations fall into an order across all three groups and rarely cross each after falling into an order. 

## Line Graphs (Antarctic Circle Groups)

Found which groups within the Antarctic Circle contain the most amount of data. Then we created a subset of the Arctic dataset in order to plot the movements of the radiation types within the group. 

In [ ]:
df_antarctic["group"].value_counts().head(5)

In [ ]:
df_antarctic82 = df_antarctic[df_antarctic["group"]==82]
df_antarctic86 = df_antarctic[df_antarctic["group"]==86]
df_antarctic1 = df_antarctic[df_antarctic["group"]==1]

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_antarctic82["timestamp"], df_antarctic82["proton0"], label="Proton",color="blue")
ax1.plot(df_antarctic82["timestamp"], df_antarctic82["electron0"], label="Electon",color="green")
ax1.plot(df_antarctic82["timestamp"], df_antarctic82["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_antarctic82["timestamp"], df_antarctic82["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

date_form = mdates.DateFormatter('%H:%M:%S')
ax1.xaxis.set_major_formatter(date_form)
ax2.xaxis.set_major_formatter(date_form)

ax1.legend(lines1 + lines2, labels1 + labels2)
plt.title("Group 82 Data (Antarctic Cirlce)", fontsize="xx-large")
fig.supxlabel("Timestamp")
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_antarctic86["timestamp"], df_antarctic86["proton0"], label="Proton",color="blue")
ax1.plot(df_antarctic86["timestamp"], df_antarctic86["electron0"], label="Electon",color="green")
ax1.plot(df_antarctic86["timestamp"], df_antarctic86["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_antarctic86["timestamp"], df_antarctic86["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2)
plt.title("Group 86 Data")
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(df_antarctic1["timestamp"], df_antarctic1["proton0"], label="Proton",color="blue")
ax1.plot(df_antarctic1["timestamp"], df_antarctic1["electron0"], label="Electon",color="green")
ax1.plot(df_antarctic1["timestamp"], df_antarctic1["xray0"], label="Xray",color="purple")

ax1.set_yscale("log")
ax1.set_ylim(1, 100000)
ax1.set_ylabel("Count (Log Scale)")

ax2 = ax1.twinx()

ax2.plot(df_antarctic1["timestamp"], df_antarctic1["ses"], label="SES",color="red")
ax2.set_ylabel("SES Count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2)
plt.title("Group 1 Data")
plt.show()

In the Antarctic Circle, we can see a trend form across all 3 of the largest groups. That trend is that proton radiation is higher than electron radiation, and electron radiation is higher than x-ray radiation. These three radiations fall into an order across all three groups and rarely cross each after falling into an order. 

# View Data in South Atlantic Anomaly

#### Find data in the South Atlantic Anomaly

In [ ]:
clean_rad[(clean_rad["lat"] < -15) & (clean_rad["lat"] > -30) & (clean_rad["lon"] < -30) & 
          (clean_rad["lon"] > -50)][["lat", "lon", "group", "timestamp", "speedmode", "in_shadow"]]

#### Find data on the same latitude as the SAA

In [ ]:
clean_rad[(clean_rad["lat"] < -15) & (clean_rad["lat"] > -30) & (clean_rad["lon"] < -110) & 
          (clean_rad["lon"] > -140)][["lat", "lon", "group", "timestamp", "speedmode", "in_shadow"]]

From these two lists, I am going to compare groups 85 and 86 because they cover the same latitudes but in different oceans, they're only about 7 hours apart, and they both have `in_shadow == 1`

## Compare Groups 85 and 86

In [ ]:
particles_path(clean_rad[clean_rad["group"] == 85])
particles_path(clean_rad[(clean_rad["group"] == 86)][66:75]) #Trimming out extra data around a spike. The trimmed data around the same geographic area was consistent with the ends of the slice

### Analysis of comparing groups 85 and 86

NOTE: They have different scales. This is a result of them being generated from separate function calls. This is important to remember

- xray0_ps stays similarly low across both groups.
    - This is surprising to me because, as I was first learning about the SAA, I expected X-ray counts to be higher here.
- We see elevated proton and electron values from group 86 at the same latitude as the SAA while xray0_ps
    - These are slightly higher than the values found in the SAA at the same time
    - This could be entirely coincidence, and it is a very small sample size, but it is good for showing that xray seems unaffected by the SAA at this altitude, even when protons and electrons spike

## Show all data points in the heart of the SAA

In [ ]:
particles_path(clean_rad[(clean_rad["lat"] < -15) & (clean_rad["lat"] > -30) & (clean_rad["lon"] < -30) & (clean_rad["lon"] > -50)])

### Analysis of data in the Heart of the SAA

There are four groups present in this graph, and they span across the entire data collection period. It's important to notice that proton0_ps is always highest, and xray0_ps is always lowest except in the last group where all three values are so similar that a change in hierarchy is unsurprisng. Again, this is a very small sample size, but the hierarchy is very interesting. The fact that xray0_ps is so low is consistent with our other analyses, so that's encouraging.

I am aware that the x-axis is very ugly. This function was built for individual groups, not for multiple groups at once

## Show similarity between groups 1 and 86

In [ ]:
particles_path(clean_rad[clean_rad["group"] == 1])
particles_path(clean_rad[clean_rad["group"] == 86])

### Quick note about group 86
Something cool I noticed during this analysis is that groups 1 and 86 follow similar paths and show fairly similar results despite being almost two months apart 
  
There's no way to test this, but I'm curious whether group 86 would have looked even more similar to group 1 if group 86 had more data points. I especially wonder this as it approaches the same latitude as the SAA